In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import itertools
import pickle

import tensorflow as tf
import tensorflow.keras as keras
import keras.layers as layers
import keras.models as models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

from wordcloud import WordCloud

from nltk.tokenize import TweetTokenizer # TweetTokenizer handles apostrophes and contractions better than word_tokenize for our use case

In [ ]:
# configure pandas ouput to show all columns and full text
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Load data

In [ ]:
# Load Dataset
data = pd.read_csv('EN_FR_data.tsv', sep='\t')
data.columns = ['EN_Sent_ID','English', 'FR_Sent_ID','French']

In [ ]:
# reduce data size to 16000 rows for feasibility of training
data = data.head(16000)

In [ ]:
data

# Preprocessing

In [ ]:
# preprocess text function
# perform lowercasing and remove punctuation that is not part of a word
def preprocess(text):
    pre_text = [word for word in TweetTokenizer().tokenize(text.lower()) if word.isalpha() or len(word) > 1]
    return " ".join(pre_text)

In [ ]:
# Apply preprocessing to both English and French columns and save to new dataframe
df_clean = data.copy()
df_clean['English'] = df_clean['English'].apply(preprocess)
df_clean['French'] = df_clean['French'].apply(preprocess)

In [ ]:
df_clean

In [ ]:
# save cleaned data to new csv file
df_clean.to_csv("cleaned_data.csv", index=False, encoding='utf-8-sig')

In [ ]:
# visualize word frequency using wordclouds
english_text = " ".join(df_clean['English'])
french_text = " ".join(df_clean['French'])

english_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(english_text)
french_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(french_text)

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.imshow(english_wordcloud)
plt.title('English Word Cloud')

plt.subplot(1, 2, 2)
plt.imshow(french_wordcloud)
plt.title('French Word Cloud')

In [ ]:
# convert dataframe to suitable format for training
# english and french sentences under one column with class labels
df_en = pd.DataFrame({
    "text": df_clean["English"],
    "label": "EN"
})

df_fr = pd.DataFrame({
    "text": df_clean["French"],
    "label": "FR"
})

df_train = pd.concat([df_en, df_fr]).sample(frac=1,random_state=42).reset_index(drop=True)

In [ ]:
df_train

In [ ]:
# define x and y for training
x= df_train['text']
y = df_train['label']

In [ ]:
# define function to convert sentence labels to token labels for token-level classification (needed for multilabel token classification)
def sentence_to_token_labels(sentences, labels):
    x_tokens = []
    y_tokens = []

    for sent, label in zip(sentences, labels):
        # white space split is enough as text is already preprocessed
        words = sent.split()

        x_tokens.append(words)
        y_tokens.append([label] * len(words))  # repeat sentence label for each token in the sentence (convert to word/token level labels)

    return x_tokens, y_tokens

In [ ]:
# use function to get token-level labels for training
x_tokens, y_tokens = sentence_to_token_labels(x, y)

In [ ]:
df_train["x_tokens"] = x_tokens
df_train["y_tokens"] = y_tokens
df_train

In [ ]:
# save training data to new csv file
df_train.drop(['text', 'label'], axis=1).to_csv("training_data.csv", index=False, encoding='utf-8-sig')

# Vectorization

In [ ]:
# get max sentence length stats to help determine max_len for padding
lengths = df_train["x_tokens"].apply(len)

print("Max length:", lengths.max())
print("Min length:", lengths.min())
print("Mean length:", lengths.mean())
print("Median length:", lengths.median())
print("95 percentile length:", lengths.quantile(0.95))

In [ ]:
Q1 = lengths.quantile(0.25)
Q3 = lengths.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

In [ ]:
outliers = df_train[(lengths < lower_bound) | (lengths > upper_bound)]

print("Number of outlier sentences:", len(outliers))
outliers[["x_tokens"]].head()

In [ ]:
# define max sentence length as 95 percentile since outliers are present with much higher lengths than the rest of the data
max_len = int(lengths.quantile(0.95))
max_len

In [ ]:
# apply label encoding to class labels using manual labels
labels = {
    "PAD": 0,
    "EN": 1,
    "FR": 2
}

# number of classes is 3 (PAD, EN, FR)
num_classes = len(labels)

def encode_labels(label_sequences):
    # for each sequence of language tags, convert each tag to its corresponding integer label using the labels dictionary
    # then pad the sentences to ensure uniform length
    encoded = [[labels[tag] for tag in seq] for seq in label_sequences]
    return pad_sequences(encoded, maxlen=max_len, padding='post')

y_enc = encode_labels(y_tokens)

In [ ]:
print(y_enc.shape)

In [ ]:
df_train["y_enc"] = list(y_enc)
df_train

In [ ]:
# perform vectorization for x using Keras Tokenizer and pad sequences to ensure uniform input length
tokenizer = Tokenizer(oov_token="<OOV>")

def tokenize(x):
    # recreate x into String format for tokenization since Keras Tokenizer expects text input
    x_flat = [" ".join(tokens) for tokens in x]

    tokenizer.fit_on_texts(x_flat)

    # convert text to sequences of integers based on the fitted tokenizer
    x_seq = tokenizer.texts_to_sequences(x_flat)

    # pad sequences to ensure uniform input length
    x_pad = pad_sequences(x_seq, maxlen=max_len, padding='post')

    vocab_size = len(tokenizer.word_index) + 1

    return x_pad, vocab_size

In [ ]:
# apply function
x_pad, vocab_size = tokenize(x_tokens)

# Train Test Split

In [ ]:
# split into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x_pad, y_enc, train_size=0.8, random_state=42)

# Model Definition and Training (Mono-Language)

In [ ]:
model = models.Sequential([
    # Embedding layers converts integer token sequences into dense vector representation
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True   # tells model to ignore padding (0)
    ),

    layers.SimpleRNN(64, 
                     return_sequences=True), # we need sequences output for token-level classification

    layers.TimeDistributed( # Time Distributed applies the same Dense layer to each time step (token) in the sequence to get token-level predictions
        layers.Dense(num_classes, activation="softmax"))
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# set sample weights to 0 for padding tokens (label 0) and 1 for non-padding tokens to prevent model from learning from padding during training
sample_weight = (y_train != 0).astype(float)

In [ ]:
# train basic RNN model as baseline (mono-language)
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    sample_weight=sample_weight,
    epochs=3,
    batch_size=32
)

In [ ]:
model.summary()

In [ ]:
preds = model.predict(x_test)
pred_labels = preds.argmax(axis=-1)

In [ ]:
df_preds = pd.DataFrame()
df_preds["y_test"] = list(y_test)
df_preds["y_pred"] = list(pred_labels)
df_preds

# Evaluation (Mono-Language)

In [ ]:
train_loss, train_acc = model.evaluate(x_train,y_train,verbose=2)
print(f"Training Accuracy: {train_acc}")

In [ ]:
test_loss, test_acc = model.evaluate(x_test,y_test,verbose=2)
print(f"Training Accuracy: {train_acc}")

In [ ]:
plt.plot(history.history['accuracy'], label="Test Accuracy")
plt.plot(history.history['val_accuracy'], label="Validation Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epochs")
plt.legend()
plt.show()

# Code-mixed synthetic data + multilanguage model

In [ ]:
def create_frenglish(row):
    en_words = row.iloc[1].split()
    fr_words = row.iloc[3].split()

    mixed_sentence = []
    labels = []

    # Take the lowest length between french or english sentence to avoid repeating words
    max_len = min(len(en_words), len(fr_words))

    for i in range(max_len):
        use_en = random.choice([True, False])

        if use_en and i < len(en_words):
            mixed_sentence.append(en_words[i])
            labels.append("EN")

        elif i < len(fr_words):
            mixed_sentence.append(fr_words[i])
            labels.append("FR")

        elif i < len(en_words):  # fallback
            mixed_sentence.append(en_words[i])
            labels.append("EN")

    return mixed_sentence, labels

In [ ]:
x_tokens_mixed = []
y_tokens_mixed = []

for _, row in df_clean.iterrows():
    words, word_labels = create_frenglish(row)
    
    x_tokens_mixed.append(words)
    y_tokens_mixed.append(word_labels)

In [ ]:
df_mixed = df_clean.drop(columns=["EN_Sent_ID","FR_Sent_ID"])
df_mixed["Frenglish"] = x_tokens_mixed
df_mixed["Labels"] = y_tokens_mixed
df_mixed

In [ ]:
# save code-mixed dataset
df_mixed.to_csv("code_mixed_data.csv", index=False, encoding='utf-8-sig')

In [ ]:
# define x and y for code-mixed training using already created token and label lists
x_mixed = x_tokens_mixed
y_mixed = y_tokens_mixed


In [ ]:
y_enc_mixed = encode_labels(y_mixed)
print(y_enc_mixed.shape)

In [ ]:
# perform vectorization for x using Tokenizer and pad sequences

x_pad_mixed, vocab_size_mixed = tokenize(x_mixed)

print(x_pad_mixed.shape)
print(vocab_size_mixed)


In [ ]:
# split into training and testing sets
x_train_mixed, x_test_mixed, y_train_mixed, y_test_mixed = train_test_split(
    x_pad_mixed, y_enc_mixed, train_size=0.8, random_state=42
)


In [ ]:
# define SimpleRNN model for code-mixed data
model_mixed = models.Sequential([
    layers.Embedding(
        input_dim=vocab_size_mixed,
        output_dim=64,
        mask_zero=True
    ),

    layers.SimpleRNN(64,return_sequences=True),

    layers.TimeDistributed(layers.Dense(num_classes, activation="softmax"))
])


In [ ]:
model_mixed.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
sample_weight_mixed = (y_train_mixed != 0).astype(float)

In [ ]:
# train basic RNN model as baseline for code-mixed data
history_mixed = model_mixed.fit(
    x_train_mixed,
    y_train_mixed,
    validation_data=(x_test_mixed, y_test_mixed),
    sample_weight=sample_weight_mixed,
    epochs=3,
    batch_size=32
)


In [ ]:
model_mixed.summary()


In [ ]:
#test prediction
preds_mixed = model_mixed.predict(x_test_mixed)
pred_labels_mixed = preds_mixed.argmax(axis=-1)


In [ ]:
df_preds_mixed = pd.DataFrame()
df_preds_mixed["y_test"] = list(y_test_mixed)
df_preds_mixed["y_pred"] = list(pred_labels_mixed)
df_preds_mixed.head()


In [ ]:
# Evaluate training accuracy on code-mixed model
train_loss_mixed, train_acc_mixed = model_mixed.evaluate(x_train_mixed, y_train_mixed, verbose=2)
print(f"Training Accuracy: {train_acc_mixed}")


In [ ]:
test_loss_mixed, test_acc_mixed = model_mixed.evaluate(x_test_mixed, y_test_mixed, verbose=2)
print(f"Testing Accuracy: {test_acc_mixed}")


In [ ]:
plt.plot(history_mixed.history['accuracy'], label="Train Accuracy")
plt.plot(history_mixed.history['val_accuracy'], label="Validation Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epochs")
plt.legend()
plt.show()

# Advanced Model (LSTM) + Hyperparameter Tuning

In [ ]:

# Define function to build advanced model (LSTM) with hyperparameters for easy tuning
def build_model(vocab_size, num_classes, emb_dim=128, lstm_units=128, dropout_rate=0.3):
    model_tune = models.Sequential([
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=emb_dim,
            mask_zero=True   
        ),

        # Bidirectional LSTM can capture context from both directions which is beneficial for language identification in code-mixed sentences
        layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True)),

        # Drop some neurons during training to prevent overfitting
        layers.Dropout(dropout_rate),

        # introduce Dense layer before output to learn more complex feature repesentation
        layers.TimeDistributed(layers.Dense(64, activation="relu")),
        layers.TimeDistributed(layers.Dense(num_classes, activation="softmax"))
    ])

    model_tune.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model_tune

In [ ]:
# hyperparameters to tune
embedding_dims = [64, 128]
lstm_units = [64, 128]
dropouts = [0.2, 0.3, 0.5]
batch_sizes = [32, 64]

In [ ]:
# define early stopping callback to prevent overfitting and save time during tuning
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2, # if model doesn't improve for 2 epochs, stop training
    restore_best_weights=True
)

In [ ]:
results = []

for emb, lstm, dr, batch in itertools.product( # utilize itertools.product to iterate through all combinations of hyperparameters
    embedding_dims, lstm_units, dropouts, batch_sizes
):

    print(f"Running: emb={emb}, lstm={lstm}, dropout={dr}, batch={batch}")

    model_tune = build_model(
        vocab_size_mixed,
        num_classes,
        emb_dim=emb,
        lstm_units=lstm,
        dropout_rate=dr
    )

    history = model_tune.fit(
        x_train_mixed,
        y_train_mixed,
        sample_weight=sample_weight_mixed,
        validation_data=(x_test_mixed, y_test_mixed),
        epochs=3,   # short runs for tuning
        batch_size=batch,
        verbose=0,
        callbacks=[early_stop]
    )

    val_acc = max(history.history["val_accuracy"])
    results.append({
        "emb": emb,
        "lstm": lstm,
        "dropout": dr,
        "batch": batch,
        "val_acc": val_acc
    })

In [ ]:
# Get best model
best = max(results, key=lambda x: x["val_acc"])

# static definition can be used to skip tuning (takes a while)
# best = {'emb': 128, 'lstm': 128, 'dropout': 0.5, 'batch': 32, 'val_acc': 0.918795645236969}

print(best)

In [ ]:
model_best = build_model(
    vocab_size_mixed,
    num_classes,
    emb_dim=best["emb"],
    lstm_units=best["lstm"],
    dropout_rate=best["dropout"]
)

In [ ]:
history_best = model_best.fit(
    x_train_mixed,
    y_train_mixed,
    sample_weight=sample_weight_mixed,
    epochs=10,
    batch_size=best["batch"],
    validation_data=(x_test_mixed, y_test_mixed)
)

In [ ]:
train_loss_best, train_acc_best = model_best.evaluate(x_train_mixed, y_train_mixed, verbose=2)
print(f"Training Accuracy: {train_acc_best}")

In [ ]:
test_loss_best, test_acc_best = model_best.evaluate(x_test_mixed, y_test_mixed, verbose=2)
print(f"Testing Accuracy: {test_acc_best}")

In [ ]:
plt.plot(history_best.history['accuracy'], label="Train Accuracy")
plt.plot(history_best.history['val_accuracy'], label="Validation Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epochs")
plt.legend()
plt.show()

In [ ]:
# function to preprocess, tokenize, and predict on new sentences
def pred_sent(sent, model, tokenizer):
    sent_pre = preprocess(sent)  # preprocess the sentence using the same preprocessing function as training data
    
    # convert text to sequences of integers based on the tokenizer
    sent_seq = tokenizer.texts_to_sequences([sent_pre])  # tokenizer expects a list of sentences

    # pad sequences to ensure uniform input length
    sent_pad = pad_sequences(sent_seq, maxlen=max_len, padding='post')

    pred = model.predict(sent_pad).argmax(axis=-1)[0]

    label_to_token = {
        0: "PAD",
        1: "EN",
        2: "FR"
    }

    for i, token in enumerate(sent_pre.split()):
        if i >= max_len:  # if sentence is longer than max_len, ignore extra tokens
            break
        print(f"{token}: {label_to_token[pred[i]]}")

In [ ]:
model_best.predict(pad_sequences(tokenizer.texts_to_sequences(["hello mon ami"]), maxlen=max_len, padding='post')).shape

In [ ]:
# test on a sample code-mixed sentence
sent = "Bonjour, my hands are le cold"
pred_sent(sent, model_best, tokenizer)

# Comparison

In [ ]:
df_comp = pd.DataFrame()
df_comp["model"] = ["RNN Mono Baseline", "RNN Code-Mixed Baseline", "LSTM Tuned"]
df_comp["train accuracy"] = [train_acc, train_acc_mixed, train_acc_best]
df_comp["test accuracy"] = [test_acc, test_acc_mixed, test_acc_best]
df_comp


# Save Model

In [ ]:
model_best.save("language_model.keras")

In [ ]:

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)